# Day 7 | ILT 2: Performance in Modelling — Z-Ordering, Joins & Aggregations
### GlobalMart Data Engineering Bootcamp

| | |
|---|---|
| **Format** | Instructor-led — run live, no student setup needed |
| **Builds on** | Day 4 ILT 3 (partitioning/Z-ORDER fundamentals), Day 7 HOL 2 (`fact_sales` now exists) |
| **Leads into** | Day 7 HOL 3 — Z-ORDER vs Liquid Clustering, applied to `fact_sales` itself |
| **Duration** | 60 minutes |

### Learning Objectives
- Understand why a star-schema fact table has *different* performance concerns than a raw Bronze table
- Choose the right Z-ORDER columns for a fact table based on real query patterns, not guesswork
- Understand broadcast joins vs sort-merge joins, and when Spark picks each automatically
- Recognize partition pruning and predicate pushdown in a real query plan

You saw the *mechanics* of Z-ORDER/Liquid Clustering/caching back in Day 4 ILT 3 on a generic table. Today is about applying that knowledge specifically to the fact table your business views (built in HOL 2) hit constantly.

---
## 1. Why Fact Tables Have Different Performance Needs Than Bronze Tables

A Bronze table is usually scanned wholesale or filtered on one or two columns (`_ingested_at`, a source ID). A fact table like `fact_sales` gets queried very differently — **almost always through a JOIN to one or more dimensions, filtered on a business dimension's attribute**, like "show me sales for the Electronics category last quarter."

```sql
SELECT ... FROM fact_sales f
JOIN dim_product p ON f.Product_ID = p.product_id   -- join column
JOIN dim_date d    ON f.Time_ID = d.date_key         -- join column
WHERE p.category = 'Electronics'                     -- filter lands on the DIMENSION, not the fact
```

The columns that matter for `fact_sales`'s performance aren't arbitrary — they're exactly the **join keys** used to reach the dimensions that carry the filters.

In [ ]:
# ============================================================
# CELL 1: Look at the real query plan for exactly this kind of query.
# .explain() shows what Spark actually decided to do -- not what we assume.
# ============================================================

query_df = spark.sql("""
    SELECT p.category, SUM(f.Sales_amount) AS revenue
    FROM gbmart.gold.fact_sales f
    JOIN gbmart.gold.dim_product p ON f.Product_ID = p.product_id AND p.is_current = true
    JOIN gbmart.gold.dim_date d    ON f.Time_ID = d.date_key
    WHERE p.category = 'Electronics'
    GROUP BY p.category
""")

query_df.explain(mode="formatted")

---
## 2. Broadcast Joins vs Sort-Merge Joins

Look at the plan above for a `BroadcastHashJoin` or a `SortMergeJoin`. Spark picks automatically, based on table size:

| | Broadcast Join | Sort-Merge Join |
|---|---|---|
| When Spark picks it | One side is small (default threshold: 10MB, configurable) | Both sides are large |
| How it works | The small table is copied whole to every executor — no shuffle needed | Both tables are shuffled and sorted on the join key, then merged |
| Cost | Cheap — no network shuffle of the big table | Expensive — shuffles the big table across the cluster |
| GlobalMart example | `dim_date` (a few thousand rows), `dim_payment_method` (a handful of rows) | `fact_sales` joined to `dim_customer` at real scale |

**This is why dimension tables are kept small and denormalized in a star schema** — a small `dim_date`/`dim_payment_method` gets broadcast almost for free, every time, without you writing a single hint.

In [ ]:
# ============================================================
# CELL 2: Confirm which dimensions are broadcast-sized in practice
# ============================================================

for dim in ["dim_date", "dim_payment_method", "dim_address", "dim_product", "dim_customer"]:
    detail = spark.sql(f"DESCRIBE DETAIL gbmart.gold.{dim}").select("sizeInBytes", "numFiles").collect()[0]
    size_mb = detail["sizeInBytes"] / (1024 * 1024)
    print(f"{dim:<22} {size_mb:>8.2f} MB   ({'likely broadcast' if size_mb < 10 else 'likely sort-merge'})")

---
## 3. Choosing Z-ORDER Columns for `fact_sales` — Not a Guess

Z-ORDER co-locates rows with similar values in the chosen column(s) into the same files, so a filter on that column skips most files entirely. The right choice comes directly from real query patterns, not intuition:

| Real query pattern (from HOL 2's views) | Column filtered/joined on | Z-ORDER candidate |
|---|---|---|
| `vw_monthly_category_sales` — join to `dim_product`, group by month | `Product_ID`, `Time_ID` | Yes — both, if query volume justifies 2 columns |
| `vw_regional_sales` — join to `dim_address` | `Address_ID` | Yes, if regional queries dominate |
| Ad-hoc "find this one customer's orders" | `Customer_ID` | Only if this pattern is actually common |

**Z-ORDER isn't free** — more Z-ORDER columns means more file-rewrite cost every time `OPTIMIZE` runs, for diminishing returns per additional column (2-3 columns is the practical ceiling). Pick the columns your *actual* dashboards and views hit most, not every column that theoretically could be filtered on.

---
## 4. Partition Pruning and Predicate Pushdown, Made Visible

Two more optimizations Spark applies automatically, visible in the same query plan:

- **Predicate pushdown**: a filter like `p.category = 'Electronics'` gets pushed down to the file-scan level for `dim_product`, so Spark never even reads rows for other categories into memory.
- **Partition pruning**: if a table is partitioned by a column that appears in the `WHERE` clause (e.g. a Bronze table partitioned by ingestion date, filtered by date range), entire partition folders are skipped without opening a single file inside them.

`fact_sales` itself isn't partitioned (Day 4's guidance: don't partition a table this size on a low-cardinality column) — it relies on Z-ORDER + file statistics (data skipping) instead. Partition pruning is more relevant to the large, append-heavy Bronze tables from Day 3–4.

---
## Recap Before HOL 3

| Concept | GlobalMart application |
|---|---|
| Broadcast vs sort-merge | Small dims (`dim_date`, `dim_payment_method`) broadcast automatically |
| Z-ORDER columns | Chosen from real view query patterns: `Product_ID`, `Time_ID`, `Address_ID` |
| Predicate pushdown | Automatic — filters on dimension attributes skip unneeded rows at scan time |
| Partitioning vs Z-ORDER | `fact_sales` uses Z-ORDER, not partitioning — table isn't large/skewed enough to need both |

Next: HOL 3 actually measures the before/after on `fact_sales`.